# Drone imagery workflow (TERN STAC API)
Search imagery items, load with ODC, slice ROI, and preview RGB.


## Before you run
- Update placeholder values (`COLLECTION_ID`, dates, bounds, point coordinates) to match your data.
- Ensure auth is configured for protected assets (for example `.netrc` and/or GDAL config).
- Install optional dependencies required by this notebook's workflow (`odc-stac`, `rioxarray`, `geopandas`, plotting extras).
- Run cells from top to bottom so variables are initialized in order.


In [ ]:
from pathlib import Path

from tern_stac import TernStacClient, load_items_odc, spatial_slice, preview_raster


In [ ]:
# Fill in from your catalog values
COLLECTION_ID = "uas_dronescape_saagaw0009_20241001__imagery_multispec"
START_DATE = "2024-01-01"
END_DATE = "2024-12-31"
BANDS = ["b5", "b4", "b3"]
REGION_BOUNDS = (135.6242307, -30.6713657, 135.6290876, -30.6668747)  # (minx, miny, maxx, maxy)
REGION_BOUNDS_CRS = "EPSG:4326"


In [ ]:
Path("outputs").mkdir(exist_ok=True)

client = TernStacClient()
search = client.search(
    collections=[COLLECTION_ID],
    datetime=f"{START_DATE}/{END_DATE}",
    bbox=[REGION_BOUNDS[0], REGION_BOUNDS[1], REGION_BOUNDS[2], REGION_BOUNDS[3]],
)
items = list(search.items())
len(items)


In [ ]:
if not items:
    raise RuntimeError("No items returned. Update collection/date/bbox placeholders.")

ds = load_items_odc(
    items,
    bands=BANDS,
    crs="utm",
    groupby="solar_day",
    chunks={},
)
ds


In [ ]:
roi = spatial_slice(ds, REGION_BOUNDS, bounds_crs=REGION_BOUNDS_CRS)
roi


In [ ]:
preview_raster(
    roi,
    rgb_bands=BANDS,
    time_index=-1,
    save_path="outputs/imagery_roi_rgb_last.png",
    title="Imagery ROI RGB (last timestep)",
)
